In [ ]:

"""
ETAPA 1: Acesso à Imagem Sentinel-2 para o sudoeste do Pará
Implementação robusta garantindo que o ROI fique totalmente dentro do estado
"""

import ee
import geemap
import json

ee.Initialize(project="desafio-solved")

states = ee.FeatureCollection("FAO/GAUL/2015/level1")

para = states.filter(ee.Filter.eq('ADM1_NAME', 'Para')).geometry()

sudoeste_box = ee.Geometry.Rectangle([-58.0, -8.5, -56.0, -6.0])

roi = para.intersection(sudoeste_box)

print("Área de interesse definida (ROI totalmente dentro do Pará)")

collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate('2023-01-01', '2023-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

total_images = collection.size().getInfo()
print(f"{total_images} imagens disponíveis")

def calculate_coverage(img):
    """Calcula percentual do ROI coberto pela imagem"""
    intersection = img.geometry().intersection(roi, ee.ErrorMargin(10))
    coverage = intersection.area().divide(roi.area())
    return img.set('coverage', coverage)

collection_with_coverage = collection.map(calculate_coverage)

best_image = collection_with_coverage.sort('coverage', False).first()
coverage_value = best_image.get('coverage').getInfo()

print(f"\nIMAGEM SELECIONADA:")
print(f"ID: {best_image.id().getInfo()}")
print(f"Data: {best_image.date().getInfo()}")
print(f"Cobertura do ROI: {coverage_value*100:.1f}%")
print(f"Nuvens: {best_image.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()}%")

if coverage_value < 0.95:
    print("Cobertura insuficiente. Criando mosaico...")
    final_image = collection.mosaic()
    print("Mosaico criado com múltiplas imagens")
else:
    final_image = best_image
    print("Imagem única com boa cobertura")

final_image = final_image.clip(roi)
print("Imagem recortada para o ROI do sudoeste do Pará")

Map = geemap.Map()
Map.centerObject(roi, 10)
Map.addLayer(roi, {}, 'ROI - Sudoeste Pará')

vis_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000,
    'gamma': 1.4
}

Map.addLayer(final_image, vis_params, 'Sentinel-2 RGB recortada')
Map.addLayerControl()
Map

metadata = {
    'etapa': '1_acesso_imagem',
    'roi_coordinates': roi.coordinates().getInfo(),
    'total_imagens_disponiveis': total_images,
    'imagem_selecionada_id': best_image.id().getInfo(),
    'imagem_selecionada_data': str(best_image.date().getInfo()),
    'cobertura_percentual': coverage_value * 100,
    'nuvens_percentual': best_image.get('CLOUDY_PIXEL_PERCENTAGE').getInfo(),
    'precisou_mosaico': coverage_value < 0.95
}

with open('etapa1_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print("\nEtapa 1 concluída Metadata salva em 'etapa1_metadata.json'")
Map

✅ Área de interesse definida (ROI totalmente dentro do Pará)
✅ 445 imagens disponíveis

📸 IMAGEM SELECIONADA:
ID: 20230330T140711_20230330T141324_T21MVN
Data: {'type': 'Date', 'value': 1680185669497}
Cobertura do ROI: 20.6%
Nuvens: 13.126984%
Cobertura insuficiente. Criando mosaico...
Mosaico criado com múltiplas imagens
✅ Imagem recortada para o ROI do sudoeste do Pará

Etapa 1 concluída Metadata salva em 'etapa1_metadata.json'


Map(center=[-7.21096138882846, -56.962456865658595], controls=(WidgetControl(options=['position', 'transparent…